# Setup

### Install libraries

In [49]:
import glob
import os
from dataretrieval import waterdata # USGS package dataretrieval to get daily discharge time series at Lowman
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from scipy.stats import linregress

### Parameter and output location selection

Select the data of interest

In [50]:
# path to save figures
fig_path = 'outputs/figures'
variable_name = 'TSS' # change this if you want DOC, FIELD_DATA or TSS. For full dataset, use 'ALL'

# path to data
data_path = 'data/water_samples_QC'
csv_files = glob.glob(os.path.join(data_path,f'*_{variable_name}*.csv'))
data = pd.read_csv(csv_files[0])

Set lowman USGS gage inputs

In [51]:
## Set parameters needed to retrieve streamflow data
start_date1 = '2024-10-01' # Water Year 2025 start

parameter_code = '00060'
end_date2 = '2026-08-07'
stat_id = '00003'

# SF Payette River at Lowman
gages = ['13235000']
gage_names = ['SF Payette River at Lowman']
my_gage = gages[0]

# Function Definition

Getting streamflow data from USGS 

In [52]:
def get_streamflow_data(
    gage: str,
    start_date: str,
    end_date: str,
    parameter_code: str,
    stats_id: str = '00003',
    convert_values: bool = False
) -> pd.DataFrame:
    """Fetches a time series of USGS streamgage observations.
    Requires the `dataretrieval` package maintained by the U.S. Geological Survey.

    Args:
        gage (str): The numeric code for an individual stream gage.
        start_date (str): The starting date of the time series in 'YYYY-MM-DD' format.
        end_date (str): The ending date of the time series in 'YYYY-MM-DD' format.
        parameter_code (str): The parameter code for the measurement of interest (e.g., '00060' for streamflow).
        stats_id (str, optional): The statistical operation resquested. Defaults to '00003' (mean).
        convert_values (bool, optional): Option to convert discharge values
            from cfs to cms. Defaults to False.

    Returns:
        pd.DataFrame: A DataFrame containing dates (str) and discharge values (float64)
        in either cfs or cms.
    """

    # Ensure the gage has the required OGC prefix
    location_id = gage if gage.startswith('USGS-') else f'USGS-{gage}'
    # Format the time as an ISO 8601 interval string
    time_interval = f"{start_date}/{end_date}"

    # Call the updated API
    daily_streamflow = waterdata.get_daily(
        monitoring_location_id=location_id,
        parameter_code=parameter_code,
        statistic_id=stats_id,
        time=time_interval
    )
    if stats_id == '00003':
        stat = "daily mean"
    else:
        stat = ""

    print(f"Pulled {len(daily_streamflow[0])} {stat} discharge values for gage {gage}")
    dt_output = daily_streamflow[0]['time']

    # Dynamically extract values based on the requested parameter code
    column_name = "value"  # The column containing the discharge values
    q_output = daily_streamflow[0][column_name].values.astype(float) 

    # Factor for converting cfs to cms
    cfs_to_cms = 35.3147

    if convert_values:
        output_data = {
            'date': dt_output,
            'q_cms': q_output / cfs_to_cms
        }
    else:
        output_data = {
            'date': dt_output,
            'q_cfs': q_output
        }

    return pd.DataFrame(output_data)

Merge discharge and concentration data

In [53]:
def merge_with_discharge(sed_df, discharge_df, date_col='date'):
    """
    Merge sediment sample data with discharge time series using the date column.
    Ensures both columns are datetime and aligns on daily dates.
    """
    # Standardize date formats
    sed_df[date_col] = pd.to_datetime(sed_df[date_col], errors='coerce')
    discharge_df[date_col] = pd.to_datetime(discharge_df[date_col], errors='coerce')

    # Drop rows with invalid or missing dates
    sed_df = sed_df.dropna(subset=[date_col])
    discharge_df = discharge_df.dropna(subset=[date_col])

    # Optional: if discharge_df has a datetime index, reset it
    if discharge_df.index.name == date_col:
        discharge_df = discharge_df.reset_index()

    # Merge on the date column
    merged = pd.merge(sed_df, discharge_df, on=date_col, how='left')

    return merged

Add seasonal descriptors

In [54]:
def add_season_column(df, date_col='date'):
    """
        Add a sesonal grouping column based on month of sampling
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    month = df[date_col].dt.month

    df['season'] = np.select(
        [
            month.isin([10, 11, 12]),
            month.isin([1, 2, 3]),
            month.isin([4, 5, 6]),
            month.isin([7, 8, 9]),
        ],
        ['OND', 'JFM', 'AMJ', 'JAS'],
        default='Unknown'
    )

    return df

Regression line 

In [55]:
def add_regression_line(subset, ax, q_col, y_col, color, label):
    if len(subset) < 3 or subset[q_col].nunique() < 2:
        return

    x = np.log10(subset[q_col].to_numpy())
    y = np.log10(subset[y_col].to_numpy())

    slope, intercept, r, p, stderr = linregress(x, y)

    if not np.isfinite(p):
        return

    q_fit = np.logspace(
        np.log10(subset[q_col].min()),
        np.log10(subset[q_col].max()),
        100
    )

    y_fit = 10 ** intercept * q_fit ** slope

    ax.plot(
        q_fit,
        y_fit,
        color=color,
        linewidth=1,
        linestyle='--',
        alpha=0.7,
        label=f"{label}: R²={r**2:.2f}, p={p:.3f}"
    )

Plot rating curve

In [56]:
def plot_rating_curve_split_date(df, site_name, split_date, site_col='site',
                                q_col='q_cfs', y_col='', date_col='date',
                                figsize=(10, 7), y_ax_label='',
                                log_y_ax=False, label_before='Before', label_after='After',
                                save_fig=False,dir_path=None):
    """
    Plot y vs discharge for a SINGLE SITE, with data split into two subsets by date.
    Each subset gets its own regression line and legend.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain site, discharge, y-variable, and date columns.
    site_name : str
        Name of the site to plot.
    split_date : str or datetime
        Date to split the dataset (e.g., '2025-07-01').
    site_col : str
        Column name for site identifier.
    q_col : str
        Column name for discharge values.
    y_col : str
        Column name for y-axis variable (e.g., 'NPOC_conc', 'DIC_conc').
    date_col : str
        Column name for the sampling date.
    figsize : tuple
        Figure size.
    y_ax_label : str
        Label for y-axis.
    log_y_ax : bool
        Whether to use log scale for y-axis.
    label_before : str
        Label for the "before" subset.
    label_after : str
        Label for the "after" subset.
    """

    # Filter data for the site
    site_df = df[df[site_col] == site_name].copy()
    site_df = site_df.dropna(subset=[q_col, y_col, date_col])
    site_df = site_df[(site_df[q_col] > 0) & (site_df[y_col] > 0)]

    if site_df.empty:
        print(f"No valid data for site {site_name}")
        return

    # Convert dates to datetime
    site_df[date_col] = pd.to_datetime(site_df[date_col])
    split_date = pd.to_datetime(split_date)

    # Split into before and after
    df_before = site_df[site_df[date_col] < split_date]
    df_after = site_df[site_df[date_col] >= split_date]

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    colors = {'before': 'tab:blue', 'after': 'tab:orange'}

    # Plot before subset
    if not df_before.empty:
        ax.scatter(df_before[q_col], df_before[y_col],
                    color=colors['before'], s=80, edgecolor='black', linewidth=1,
                    label=f'{label_before} (n={len(df_before)})', alpha=0.7)
        # Fit regression for before
        add_regression_line(
            df_before, ax, q_col, y_col,
            colors["before"], label_before
        )

    # Plot after subset
    if not df_after.empty:
        ax.scatter(df_after[q_col], df_after[y_col],
                    color=colors['after'], s=80, edgecolor='black', linewidth=1,
                    label=f'{label_after} (n={len(df_after)})', alpha=0.7)
        # Fit regression for after
        add_regression_line(
            df_after, ax, q_col, y_col,
            colors["after"], label_after
        )

    # Log scales
    plt.xscale('log')
    if log_y_ax:
        ax.set_yscale("log")
        ax.set_ylim(0.1, site_df[y_col].max()*1.3)
    else:
        ax.set_ylim(0, site_df[y_col].max()*1.3)

    plt.xlabel('Discharge (cfs)',fontsize=8)
    plt.ylabel(y_ax_label, fontsize=8)
    plt.title(f'{site_name}',fontsize=10)
    ax.legend(fontsize=6, loc="upper center", bbox_to_anchor=(0.5, -0.18),
    ncol=2, frameon=False)
    plt.grid(True, which='both', ls='--', alpha=0.3)
    plt.tight_layout(rect=[0, 0.12, 1, 1])

    if save_fig:
        plt.savefig(os.path.join(dir_path, f'{site_name}.png'), dpi=300, bbox_inches='tight')
        plt.close()
        print(f'saving {site_name}')
    else:
        plt.show()


Plot rating curves for different years 

In [57]:
def plot_rating_curve_year(df, site_name, site_col='site',
    q_col='q_cfs', y_col='', date_col='date', year_type='calendar',
    figsize=(10, 7), y_ax_label='', log_y_ax=False, 
    years=None, save_fig=False, dir_path=None):
    """
    Plot a rating curve for one site with samples grouped by calendar year.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing site, discharge, response variable, and date columns.
    site_name : str
        Name of the site to plot.
    site_col : str
        Column containing site identifiers.
    q_col : str
        Column containing discharge values.
    y_col : str
        Column containing the response variable.
    date_col : str
        Column containing sampling dates.
    figsize : tuple
        Figure size.
    y_ax_label : str
        Label for the y-axis.
    log_y_ax : bool
        Whether to use a logarithmic y-axis.
    years : list, optional
        Years to plot. If None, all available years are plotted.
    save_fig : bool
        Whether to save the figure.
    dir_path : str, optional
        Directory where the figure should be saved.
    """

    # Filter and prepare data for the specified site
    site_df = df[df[site_col] == site_name].copy()
    site_df[date_col] = pd.to_datetime(site_df[date_col], errors='coerce')
    site_df = site_df.dropna(subset=[q_col, y_col, date_col])
    site_df = site_df[(site_df[q_col] > 0) & (site_df[y_col] > 0)]

    if site_df.empty:
        print(f"No valid data for site {site_name}")
        return

    if year_type == 'calendar':
        site_df['year'] = site_df[date_col].dt.year
    elif year_type == 'water':
        site_df['year'] = (site_df[date_col].dt.year + (site_df[date_col].dt.month >= 10).astype(int))

    if years is None:
        years = sorted(site_df['year'].unique())
    else:
        years = sorted(years)
        site_df = site_df[site_df['year'].isin(years)]

    if site_df.empty:
        print(f"No valid data for site {site_name} in the requested years")
        return

    fig, ax = plt.subplots(figsize=figsize)
    colors = plt.cm.viridis(np.linspace(0, 1, len(years)))
    for year, color in zip(years, colors):
        year_df = site_df[site_df['year'] == year]

        if year_df.empty:
            continue

        ax.scatter(year_df[q_col], year_df[y_col],
            color=color, s=80, edgecolor='black', linewidth=1,
            alpha=0.9, label=f'{year} (n={len(year_df)})')

        add_regression_line(year_df, ax, q_col,
                            y_col, color, str(year))
    ax.set_xscale('log')

    if log_y_ax:
        ax.set_yscale('log')
        ax.set_ylim(max(0.1, site_df[y_col].min() / 1.5),
                    site_df[y_col].max() * 1.3)
    else:
        ax.set_ylim(0, site_df[y_col].max() * 1.3)

    ax.set_xlabel('Discharge (cfs)', fontsize=8)
    ax.set_ylabel(y_ax_label, fontsize=8)
    ax.set_title(site_name, fontsize=10)

    ax.legend(fontsize=6, loc='upper center', bbox_to_anchor=(0.5, -0.18),
        ncol=2, frameon=False)

    ax.grid(True, which='both', linestyle='--', alpha=0.3)
    plt.tight_layout(rect=[0, 0.12, 1, 1])

    if save_fig:
        plt.savefig(os.path.join(dir_path, f'{site_name}_by_year.png'), dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Saving {site_name}_by_year.png')
    else:
        plt.show()

Plot gradient scatter 

In [58]:
def plot_rating_curve_gradient(df, site_name, site_col='site',
    q_col='q_cfs', y_col='', date_col='date', figsize=(5, 4),
    y_ax_label='', log_y_ax=False, save_fig=False, dir_path=None):
    """
    Plot a rating curve with point colors representing sample date.

    Older samples are darker purple and newer samples are brighter yellow
    using the viridis colormap.
    """

    site_df = df[df[site_col] == site_name].copy()
    site_df[date_col] = pd.to_datetime(site_df[date_col], errors='coerce')
    site_df = site_df.dropna(subset=[q_col, y_col, date_col])
    site_df = site_df[(site_df[q_col] > 0) & (site_df[y_col] > 0)]

    if site_df.empty:
        print(f"No valid data for site {site_name}")
        return

    # Convert dates to Matplotlib's numeric date format
    date_numbers = mdates.date2num(site_df[date_col])

    fig, ax = plt.subplots(figsize=figsize)
    scatter = ax.scatter(site_df[q_col], site_df[y_col],
        c=date_numbers, cmap='viridis', s=80,
        edgecolor='black', linewidth=1, alpha=0.9)

    ax.set_xscale('log')

    if log_y_ax:
        ax.set_yscale('log')
        ax.set_ylim(max(0.1, site_df[y_col].min() / 1.5),
            site_df[y_col].max() * 1.3)
    else:
        ax.set_ylim(0, site_df[y_col].max() * 1.3)

    # Add a date-formatted colorbar
    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    colorbar.set_label('Sample date', fontsize=8)

    ax.set_xlabel('Discharge (cfs)', fontsize=8)
    ax.set_ylabel(y_ax_label, fontsize=8)
    ax.set_title(site_name, fontsize=10)

    ax.grid(True, which='both', linestyle='--', alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()

    if save_fig:
        plt.savefig(os.path.join(dir_path, f'{site_name}_date_color.png'), dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Saving {site_name}_date_color.png')
    else:
        plt.show()

# Plotting Rating Curves

Importing streamflow data

In [59]:
sfp_streamflow = get_streamflow_data(
    gage=my_gage,
    start_date=start_date1,
    end_date=end_date2,
    stats_id=stat_id,
    parameter_code=parameter_code, 
    convert_values=False
)

sfp_streamflow.describe()

Retrieving: daily · 1 page · 674 rows


Pulled 674 daily mean discharge values for gage 13235000


,date,q_cfs
count,674,674.000000
mean,2025-09-02 12:08:32.759643904,838.316024
min,2024-10-01 00:00:00,222.000000
25%,2025-03-18 06:00:00,358.000000
50%,2025-09-02 12:00:00,481.000000
75%,2026-02-17 18:00:00,1110.000000
max,2026-08-07 00:00:00,4200.000000
std,NaN,749.917416


Merge discharge and concentration

In [60]:
# only keep these sites: 
sites_to_keep = ['SFP_Wapiti', 'Bear_Cr', 'SH_Cr', 'Camp_Cr', 'Canyon_Cr_SFP', 'SFP_Canyon_Cr', 'Fox_Cr', 
                    'SFP_Wolf_Cr', 'HP', 'Avalanche_Cr', 'GO_Cr', 'SFP_Chap', 'Warm_Sp', 'SFP_BJ', 'BJ_Cr', 'SFP_DW', 'SFP_Alder_Cr']

merged_data = merge_with_discharge(data, sfp_streamflow)
data_seasonal = add_season_column(merged_data, date_col='date')
data_seasonal = data_seasonal[data_seasonal['site'].isin(sites_to_keep)]
site_names = data_seasonal['site'].unique()

In [61]:
site_names

array(['Warm_Sp', 'SH_Cr', 'Canyon_Cr_SFP', 'BJ_Cr', 'SFP_Alder_Cr',
       'SFP_DW', 'SFP_BJ', 'SFP_Chap', 'SFP_Wolf_Cr', 'Fox_Cr',
       'SFP_Canyon_Cr', 'Camp_Cr', 'SFP_Wapiti', 'Avalanche_Cr', 'HP',
       'GO_Cr', 'Bear_Cr'], dtype=object)

In [62]:
data_seasonal.head()

,sample_num,date,site,tin_num,tin_filter_weight,bottle_tare,bottle_full,tin_filter_sed_weight,tin_filter_sed_ashed_weight,sed_weight_mg,TSS_conc_mgL-1,susp_solids_mg,susp_sed_mg,q_cfs,season
0,1,2024-10-11,Warm_Sp,19,1.208,88.57,1135.00,1.208,1.205,0.0,1.04643,0.0,3.0,299.0,OND
1,2,2024-10-11,SH_Cr,26,1.214,88.54,1133.87,1.234,-9999.000,20.0,1.04533,20.0,-9999.0,299.0,OND
2,3,2024-10-11,Canyon_Cr_SFP,25,1.232,87.50,1117.52,1.241,1.236,9.0,1.03002,9.0,5.0,299.0,OND
3,4,2024-10-11,BJ_Cr,21,1.213,87.43,1080.30,1.238,-9999.000,25.0,0.99287,25.0,-9999.0,299.0,OND
4,5,2024-10-12,SFP_Alder_Cr,16,1.228,88.35,1159.42,1.231,1.228,3.0,1.07107,3.0,3.0,297.0,OND


In [63]:
split = '2025-08-01' # change this date to vizualize recently added values
# single plots with colored by season
for i in range(0, len(site_names)):
    plot_rating_curve_split_date(
        data_seasonal,
        site_name=site_names[i],
        split_date=split,
        q_col='q_cfs',
        y_col='susp_sed_mg',
        y_ax_label='TSS (mg/L)',
        log_y_ax=True,
        figsize=(4, 4),
        label_before=f'before {split}',
        label_after=f'after {split}',
        save_fig=True,
        dir_path=fig_path
        )

saving Warm_Sp
saving SH_Cr
saving Canyon_Cr_SFP
saving BJ_Cr
saving SFP_Alder_Cr
saving SFP_DW
saving SFP_BJ
saving SFP_Chap
saving SFP_Wolf_Cr
saving Fox_Cr
saving SFP_Canyon_Cr
saving Camp_Cr
saving SFP_Wapiti
saving Avalanche_Cr
saving HP
saving GO_Cr
saving Bear_Cr


Plot rating curves by calendar year

In [64]:
fig_path = 'outputs/figures/calendar_years'
for site_name in site_names:
    plot_rating_curve_year(
        data_seasonal,
        site_name=site_name,
        q_col='q_cfs',
        y_col='susp_sed_mg',
        year_type='calendar',
        y_ax_label='TSS (mg/L)',
        log_y_ax=True,
        years=[2024, 2025, 2026],
        figsize=(4, 4),
        save_fig=True,
        dir_path=fig_path
    )

Saving Warm_Sp_by_year.png
Saving SH_Cr_by_year.png
Saving Canyon_Cr_SFP_by_year.png
Saving BJ_Cr_by_year.png
Saving SFP_Alder_Cr_by_year.png
Saving SFP_DW_by_year.png
Saving SFP_BJ_by_year.png
Saving SFP_Chap_by_year.png
Saving SFP_Wolf_Cr_by_year.png
Saving Fox_Cr_by_year.png
Saving SFP_Canyon_Cr_by_year.png
Saving Camp_Cr_by_year.png
Saving SFP_Wapiti_by_year.png
Saving Avalanche_Cr_by_year.png
Saving HP_by_year.png
Saving GO_Cr_by_year.png
Saving Bear_Cr_by_year.png


Plot rating curves by water year 

In [65]:
fig_path = 'outputs/figures/water_years'
for site_name in site_names:
    plot_rating_curve_year(
        data_seasonal,
        site_name=site_name,
        q_col='q_cfs',
        y_col='susp_sed_mg',
        year_type='water',
        y_ax_label='TSS (mg/L)',
        log_y_ax=True,
        years=[2024, 2025, 2026],
        figsize=(4, 4),
        save_fig=True,
        dir_path=fig_path
    )

Saving Warm_Sp_by_year.png
Saving SH_Cr_by_year.png
Saving Canyon_Cr_SFP_by_year.png
Saving BJ_Cr_by_year.png
Saving SFP_Alder_Cr_by_year.png
Saving SFP_DW_by_year.png
Saving SFP_BJ_by_year.png
Saving SFP_Chap_by_year.png
Saving SFP_Wolf_Cr_by_year.png
Saving Fox_Cr_by_year.png
Saving SFP_Canyon_Cr_by_year.png
Saving Camp_Cr_by_year.png
Saving SFP_Wapiti_by_year.png
Saving Avalanche_Cr_by_year.png
Saving HP_by_year.png
Saving GO_Cr_by_year.png
Saving Bear_Cr_by_year.png


Plot gradient scatter plots

In [66]:
fig_path = 'outputs/figures/gradient'
for site_name in site_names:
    plot_rating_curve_gradient(
        data_seasonal,
        site_name=site_name,
        q_col='q_cfs',
        y_col='susp_sed_mg',
        y_ax_label='TSS (mg/L)',
        log_y_ax=True,
        figsize=(6, 4),
        save_fig=True,
        dir_path=fig_path
    )

Saving Warm_Sp_date_color.png
Saving SH_Cr_date_color.png
Saving Canyon_Cr_SFP_date_color.png
Saving BJ_Cr_date_color.png
Saving SFP_Alder_Cr_date_color.png
Saving SFP_DW_date_color.png
Saving SFP_BJ_date_color.png
Saving SFP_Chap_date_color.png
Saving SFP_Wolf_Cr_date_color.png
Saving Fox_Cr_date_color.png
Saving SFP_Canyon_Cr_date_color.png
Saving Camp_Cr_date_color.png
Saving SFP_Wapiti_date_color.png
Saving Avalanche_Cr_date_color.png
Saving HP_date_color.png
Saving GO_Cr_date_color.png
Saving Bear_Cr_date_color.png
